<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/GPT_5_2_ARC_AGI_2_PerformanceTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-5.2 모델 추론 능력 테스트하기(ARC-AGI-2 데이터셋)
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## ARC-AGI-2 데이터셋 : https://github.com/arcprize/ARC-AGI-2.git

##Setting API Key ##

In [ ]:
import os

# OpenAI API Key 설정
os.environ["OPENAI_API_KEY"] = "Input Your Key"

In [ ]:
# =========================
# 1) SETUP (Colab)
# =========================
import os
from pathlib import Path

REPO_DIR = Path("ARC-AGI-2")

#ARC-AGI 데이터셋 다운
REPO_URL = "https://github.com/arcprize/ARC-AGI-2.git"

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL}

TRAIN_DIR = REPO_DIR / "data" / "training"
EVAL_DIR  = REPO_DIR / "data" / "evaluation"

print("Train dir:", TRAIN_DIR, "exists:", TRAIN_DIR.exists())
print("Eval  dir:", EVAL_DIR,  "exists:", EVAL_DIR.exists())
print("Train json files:", len(list(TRAIN_DIR.glob("*.json"))))
print("Eval  json files:", len(list(EVAL_DIR.glob("*.json"))))

In [ ]:
# =========================
# 2) LOAD TASKS
# =========================
import json
import random

def load_split_tasks(split_dir: Path) -> dict:
    """
    Returns:
      tasks: dict[str, dict]  # {task_id: task_json}
    """
    tasks = {}
    for fp in sorted(split_dir.glob("*.json")):
        task_id = fp.stem
        with open(fp, "r", encoding="utf-8") as f:
            tasks[task_id] = json.load(f)
    return tasks

tasks_train = load_split_tasks(TRAIN_DIR)
tasks_eval  = load_split_tasks(EVAL_DIR)

print("Loaded train:", len(tasks_train))
print("Loaded eval :", len(tasks_eval))

# Peek one task schema
sample_id = next(iter(tasks_train))
sample_task = tasks_train[sample_id]
print("\nSample task id:", sample_id)
print("Top-level keys:", list(sample_task.keys()))
print("train pairs:", len(sample_task.get("train", [])))
print("test  pairs:", len(sample_task.get("test", [])))
print("pair keys (train[0]):", list(sample_task["train"][0].keys()))

In [ ]:
sample_task

In [ ]:
# =========================
# 3) DATASET "구성" 요약(통계/테이블)
# =========================
import numpy as np
import pandas as pd

def grid_shape(grid):
    a = np.array(grid)
    return (a.shape[0], a.shape[1])

def summarize_split(tasks: dict, split_name: str) -> pd.DataFrame:
    rows = []
    for task_id, task in tasks.items():
        train_pairs = task.get("train", [])
        test_pairs  = task.get("test", [])

        # collect shapes
        def shapes(pairs, field):
            out = []
            for p in pairs:
                if field in p:
                    out.append(grid_shape(p[field]))
                else:
                    out.append(None)
            return out

        rows.append({
            "split": split_name,
            "task_id": task_id,
            "n_train": len(train_pairs),
            "n_test": len(test_pairs),
            "train_in_shapes": shapes(train_pairs, "input"),
            "train_out_shapes": shapes(train_pairs, "output"),
            "test_in_shapes": shapes(test_pairs, "input"),
            "test_out_shapes": shapes(test_pairs, "output"),  # 혹시 없는 경우 None
        })
    return pd.DataFrame(rows)

df_train = summarize_split(tasks_train, "training")
df_eval  = summarize_split(tasks_eval,  "evaluation")
df = pd.concat([df_train, df_eval], ignore_index=True)

display(df.head(3))
print("\nCounts by split:\n", df["split"].value_counts())

print("\nTrain n_train distribution:\n", df_train["n_train"].value_counts().sort_index())
print("\nEval  n_train distribution:\n", df_eval["n_train"].value_counts().sort_index())

# Grid size 통계(입력 기준, train+test 모두)
def collect_shapes(tasks: dict):
    hs, ws = [], []
    for task in tasks.values():
        for part in ["train", "test"]:
            for p in task.get(part, []):
                if "input" in p:
                    h, w = grid_shape(p["input"])
                    hs.append(h); ws.append(w)
    return np.array(hs), np.array(ws)

h_tr, w_tr = collect_shapes(tasks_train)
h_ev, w_ev = collect_shapes(tasks_eval)

print("\n[TRAIN input shapes] H(min/mean/max):", h_tr.min(), h_tr.mean(), h_tr.max(),
      " W(min/mean/max):", w_tr.min(), w_tr.mean(), w_tr.max())
print("[EVAL  input shapes] H(min/mean/max):", h_ev.min(), h_ev.mean(), h_ev.max(),
      " W(min/mean/max):", w_ev.min(), w_ev.mean(), w_ev.max())

In [ ]:
# =========================
# 4) VISUALIZATION (matplotlib)
# =========================
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

# 10-class discrete colormap (0~9)
cmap = plt.cm.get_cmap("tab10", 10)
norm = BoundaryNorm(np.arange(-0.5, 10.5, 1), cmap.N)

def plot_arc_grid(ax, grid, title=None, show_gridlines=True):
    arr = np.array(grid, dtype=int)
    ax.imshow(arr, cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    if show_gridlines:
        h, w = arr.shape
        ax.set_xticks(np.arange(-.5, w, 1), minor=True)
        ax.set_yticks(np.arange(-.5, h, 1), minor=True)
        ax.grid(which="minor", linewidth=0.3)
        ax.tick_params(which="both", bottom=False, left=False)

def visualize_task(task: dict, task_id: str = "", max_train=None, show_gridlines=True):
    train_pairs = task.get("train", [])
    test_pairs  = task.get("test", [])

    if max_train is not None:
        train_pairs = train_pairs[:max_train]

    # rows = train pairs + test pairs
    rows = [("train", i, p) for i, p in enumerate(train_pairs)] + \
           [("test",  i, p) for i, p in enumerate(test_pairs)]

    nrows = max(1, len(rows))
    fig, axes = plt.subplots(nrows, 2, figsize=(6, 2.6 * nrows))
    if nrows == 1:
        axes = np.array([axes])

    fig.suptitle(f"Task {task_id}  |  train={len(task.get('train', []))}, test={len(task.get('test', []))}",
                 fontsize=12, y=1.02)

    for r, (kind, idx, pair) in enumerate(rows):
        inp = pair.get("input")
        out = pair.get("output", None)  # 혹시 output이 없는 형태도 대비

        plot_arc_grid(axes[r, 0], inp, title=f"{kind}[{idx}] input", show_gridlines=show_gridlines)
        if out is None:
            axes[r, 1].text(0.5, 0.5, "(no output provided)", ha="center", va="center")
            axes[r, 1].set_axis_off()
        else:
            plot_arc_grid(axes[r, 1], out, title=f"{kind}[{idx}] output", show_gridlines=show_gridlines)

    plt.tight_layout()
    plt.show()

# 랜덤 1개 시각화 (training)
rand_id = random.choice(list(tasks_train.keys()))
visualize_task(tasks_train[rand_id], task_id=rand_id, max_train=3, show_gridlines=True)

In [ ]:
# =========================
# 5) (옵션) 위젯으로 task 선택해서 보기 (Colab)
# =========================
# 위젯이 필요 없으면 이 셀은 건너뛰셔도 됩니다.
!pip -q install "ipywidgets>=7,<9"

from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
from IPython.display import display, clear_output

split_dd = widgets.Dropdown(
    options=[("training", "training"), ("evaluation", "evaluation")],
    value="training",
    description="split:"
)

def task_options(split):
    return sorted(list(tasks_train.keys())) if split == "training" else sorted(list(tasks_eval.keys()))

task_dd = widgets.Dropdown(
    options=task_options("training"),
    value=task_options("training")[0],
    description="task:"
)

max_train_slider = widgets.IntSlider(value=3, min=1, max=10, step=1, description="max_train:")

out = widgets.Output()

def on_change_split(change):
    if change["name"] == "value":
        opts = task_options(change["new"])
        task_dd.options = opts
        task_dd.value = opts[0]

split_dd.observe(on_change_split, names="value")

def render(*_):
    with out:
        clear_output(wait=True)
        split = split_dd.value
        task_id = task_dd.value
        task = tasks_train[task_id] if split == "training" else tasks_eval[task_id]
        visualize_task(task, task_id=task_id, max_train=max_train_slider.value, show_gridlines=True)

task_dd.observe(lambda c: render(), names="value")
max_train_slider.observe(lambda c: render(), names="value")

display(widgets.VBox([split_dd, task_dd, max_train_slider, out]))
render()

In [ ]:
import time
import json
import numpy as np
import pandas as pd

from openai import OpenAI
from pydantic import BaseModel, conint
from typing import List, Optional, Dict, Any

client = OpenAI()

MODEL_4O       = "gpt-4o"
MODEL_5_2      = "gpt-5.2"
MODEL_5_2_PRO  = "gpt-5.2-pro"

# (선택) 내 계정에서 모델이 보이는지 확인
def list_available_models(prefix=None, limit=None):
    ids = [m.id for m in client.models.list().data]
    if prefix:
        ids = [x for x in ids if x.startswith(prefix)]
    ids = sorted(ids)
    return ids if limit is None else ids[:limit]

print("Has gpt-4o? ", MODEL_4O in list_available_models())
print("Has gpt-5.2? ", MODEL_5_2 in list_available_models())
print("Has gpt-5.2-pro? ", MODEL_5_2_PRO in list_available_models())

In [ ]:
# Structured Outputs용 스키마: "grid" 한 개 필드에 2D int 배열(0~9)
IntCell = conint(ge=0, le=9)

class GridPrediction(BaseModel):
    grid: List[List[IntCell]]

def build_arc_prompt(task: dict) -> str:
    """
    ARC task를 LLM에게 주기 위한 텍스트 프롬프트 생성
    - train: input/output 예시들
    - test: input 하나
    - output: test의 output grid를 grid로만 반환
    """
    train_pairs = task.get("train", [])
    test_pairs  = task.get("test", [])
    if not test_pairs:
        raise ValueError("No test pairs in task")

    # 여기선 test[0]만 평가 (원하면 루프에서 test idx를 바꿔 호출)
    test_in = test_pairs[0]["input"]

    prompt = []
    prompt.append("You are solving an ARC (Abstraction and Reasoning Corpus) task.")
    prompt.append("Grids are 2D arrays of integers 0-9 representing colors.")
    prompt.append("")
    prompt.append("Training examples (learn the rule):")
    for i, p in enumerate(train_pairs):
        prompt.append(f"- train[{i}].input = {json.dumps(p['input'])}")
        prompt.append(f"  train[{i}].output = {json.dumps(p['output'])}")
    prompt.append("")
    prompt.append("Now solve the test case:")
    prompt.append(f"- test[0].input = {json.dumps(test_in)}")
    prompt.append("")
    prompt.append("Return ONLY the predicted output grid, following the provided schema.")
    return "\n".join(prompt)

def predict_grid(model: str, task: dict, *, temperature: float = 0.0, max_retries: int = 2) -> Dict[str, Any]:
    """
    Returns dict:
      {
        "ok": bool,
        "grid": Optional[list[list[int]]],
        "error": Optional[str],
        "latency_s": float
      }
    """
    prompt = build_arc_prompt(task)

    # gpt-5.x 계열은 reasoning 파라미터를 쓰고 싶으면 아래를 켜도 됨(비교 공정성을 위해 기본은 OFF)
    extra_kwargs = {}
    # if model.startswith("gpt-5"):
    #     extra_kwargs["reasoning"] = {"effort": "high"}

    last_err = None
    for _ in range(max_retries + 1):
        t0 = time.time()
        try:
            req = dict(
                model=model,
                input=[
                    {"role": "system", "content": "Solve ARC tasks. Output must match the schema exactly."},
                    {"role": "user", "content": prompt},
                ],
                text_format=GridPrediction,
                **extra_kwargs,
            )
            # gpt-5.2-pro 등은 temperature를 지원하지 않음
            if not str(model).endswith("-pro"):
                req["temperature"] = temperature

            resp = client.responses.parse(**req)
            latency = time.time() - t0
            pred = resp.output_parsed  # pydantic object
            return {"ok": True, "grid": pred.grid, "error": None, "latency_s": latency}
        except Exception as e:
            latency = time.time() - t0
            last_err = f"{type(e).__name__}: {e}"
            # 재시도
            continue

    return {"ok": False, "grid": None, "error": last_err, "latency_s": float("nan")}

In [ ]:
import os
import json
import random

def has_test_output(task: dict) -> bool:
    t = task.get("test", [])
    return bool(t) and isinstance(t[0], dict) and ("output" in t[0])

def exact_match(a, b) -> bool:
    return a == b

def pixel_accuracy(pred, gt) -> float:
    # shape 다르면 0 처리(원하면 resize/overlap 방식으로 바꿀 수 있음)
    pa = np.array(pred, dtype=int)
    ga = np.array(gt, dtype=int)
    if pa.shape != ga.shape:
        return 0.0
    return float((pa == ga).mean())

RESULTS_PATH = "arc_results.jsonl"  # 처음부터 결과(input/gt/pred 포함)를 누적 저장

def load_results_jsonl(path: str = RESULTS_PATH):
    rows = []
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
    return rows

def append_result_jsonl(row: dict, path: str = RESULTS_PATH):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def run_models_on_samples_and_save(
    tasks: dict,
    *,
    models: list,
    sample_n: int = 10,
    seed: int = 0,
    temperature: float = 0.0,
    results_path: str = RESULTS_PATH,
):
    """처음부터 input/gt/pred까지 저장하면서 실행 (중간에 끊겨도 이어서 가능)"""

    random.seed(seed)

    candidate_ids = [tid for tid, t in tasks.items() if has_test_output(t)]
    if len(candidate_ids) == 0:
        raise RuntimeError("No tasks with test output found in this split.")

    chosen = random.sample(candidate_ids, k=min(sample_n, len(candidate_ids)))

    existing = load_results_jsonl(results_path)
    done_keys = set((r.get("task_id"), r.get("model")) for r in existing)

    for idx, task_id in enumerate(chosen, 1):
        task = tasks[task_id]
        test0 = task["test"][0]
        test_in = test0["input"]
        gt = test0.get("output")

        for model in models:
            key = (task_id, model)
            if key in done_keys:
                continue

            res = predict_grid(model, task, temperature=temperature)

            row = {
                "task_id": task_id,
                "model": model,
                "ok": bool(res.get("ok", False)),
                "latency_s": res.get("latency_s", None),
                "error": res.get("error", None),
                "input": test_in,
                "gt": gt,
                "pred_grid": res.get("grid", None),
            }

            if row["ok"] and (gt is not None) and (row["pred_grid"] is not None):
                row["exact"] = exact_match(row["pred_grid"], gt)
                row["pixel_acc"] = pixel_accuracy(row["pred_grid"], gt)
            else:
                row["exact"] = False
                row["pixel_acc"] = 0.0

            append_result_jsonl(row, results_path)
            done_keys.add(key)

        print(f"[{idx}/{len(chosen)}] done: {task_id}")

    df_results = pd.DataFrame(load_results_jsonl(results_path))
    return df_results


def build_df_vis_from_results(df_results: pd.DataFrame, model_4o: str, model_5: str, model_5pro: str) -> pd.DataFrame:
    """df_results(long: task_id x model)을 df_vis(wide: task_id 1행)로 변환.

    주의: input/gt/pred_grid는 list(2D)이므로 merge key로 쓰면 pandas가 해시 불가(TypeError) ->
    task_id로만 조인하고, input/gt는 task_id별 first()로 붙인다.
    """

    # 중복 실행으로 같은 (task_id, model) 행이 여러 개 있을 수 있으니 마지막 1개만 사용
    df = df_results.copy()
    df = df[df["model"].isin([model_4o, model_5, model_5pro])].copy()
    df = df.drop_duplicates(subset=["task_id", "model"], keep="last")

    # task_id별 input/gt는 동일하다고 가정하고 첫 값 사용
    base = df.groupby("task_id", as_index=False).agg({"input": "first", "gt": "first"})

    d4 = df[df["model"] == model_4o][["task_id", "ok", "error", "pred_grid"]].copy()
    d4 = d4.rename(
        columns={
            "ok": "gpt-4o_ok",
            "error": "gpt-4o_error",
            "pred_grid": "gpt-4o_pred",
        }
    )

    d5 = df[df["model"] == model_5][["task_id", "ok", "error", "pred_grid"]].copy()
    d5 = d5.rename(
        columns={
            "ok": "gpt-5_ok",
            "error": "gpt-5_error",
            "pred_grid": "gpt-5_pred",
        }
    )

    d5p = df[df["model"] == model_5pro][["task_id", "ok", "error", "pred_grid"]].copy()
    d5p = d5p.rename(
        columns={
            "ok": "gpt-5-pro_ok",
            "error": "gpt-5-pro_error",
            "pred_grid": "gpt-5-pro_pred",
        }
    )

    df_vis = (
        base
        .merge(d4, on="task_id", how="left")
        .merge(d5, on="task_id", how="left")
        .merge(d5p, on="task_id", how="left")
    )
    return df_vis.sort_values("task_id").reset_index(drop=True)


df_results = run_models_on_samples_and_save(
    tasks_train,
    models=[MODEL_4O, MODEL_5_2, MODEL_5_2_PRO],
    sample_n=4,
    seed=42,
    temperature=0.0,
    results_path=RESULTS_PATH,
)

# 기존 집계 로직과 호환되게 df_scores도 만들어 둠
cols_scores = ["task_id", "model", "ok", "latency_s", "error", "exact", "pixel_acc"]
df_scores = df_results[cols_scores].copy()

# 시각화용 1행/1task 형태
MODEL_5_FOR_VIS = MODEL_5_2
df_vis = build_df_vis_from_results(
    df_results,
    model_4o=MODEL_4O,
    model_5=MODEL_5_FOR_VIS,
    model_5pro=MODEL_5_2_PRO,
)

display(df_scores)
print("Saved to:", RESULTS_PATH)
print("df_scores rows:", len(df_scores), " | df_vis rows:", len(df_vis))

In [ ]:
# 모델별 집계
summary = (
    df_scores.groupby("model")
    .agg(
        n=("task_id", "count"),
        ok_rate=("ok", "mean"),
        exact_acc=("exact", "mean"),
        mean_pixel_acc=("pixel_acc", "mean"),
        median_latency_s=("latency_s", "median"),
        n_errors=("ok", lambda s: int((~s).sum())),
    )
    .reset_index()
)

display(summary)

In [ ]:
# =========================
# 6) VIS: input/gt vs (gpt-4o / gpt-5 / gpt-5-pro) preds
# =========================
import textwrap
from typing import Optional


def _plot_text_panel_3(ax, title: str, text: str):
    ax.set_axis_off()
    ax.set_title(title, fontsize=10)
    ax.text(
        0.5,
        0.5,
        textwrap.fill(text or "", width=60),
        ha="center",
        va="center",
        fontsize=9,
    )


def visualize_input_gt_3models(
    df_vis: pd.DataFrame,
    *,
    task_id: Optional[str] = None,
    idx: int = 0,
    show_gridlines: bool = True,
):
    """df_vis의 한 row를 3모델 비교로 시각화: input / gt / (gpt-4o, gpt-5, gpt-5-pro) pred"""

    if task_id is not None:
        hit = df_vis[df_vis["task_id"] == task_id]
        if len(hit) == 0:
            raise KeyError(f"task_id not found in df_vis: {task_id}")
        row = hit.iloc[0]
    else:
        row = df_vis.iloc[int(idx)]

    tid = row["task_id"]
    inp = row.get("input")
    gt = row.get("gt")

    preds = [
        ("gpt-4o", bool(row.get("gpt-4o_ok", False)), row.get("gpt-4o_error"), row.get("gpt-4o_pred")),
        ("gpt-5", bool(row.get("gpt-5_ok", False)), row.get("gpt-5_error"), row.get("gpt-5_pred")),
        ("gpt-5-pro", bool(row.get("gpt-5-pro_ok", False)), row.get("gpt-5-pro_error"), row.get("gpt-5-pro_pred")),
    ]

    if gt is None:
        fig, axes = plt.subplots(1, 1 + len(preds), figsize=(4 * (1 + len(preds)), 3.6))
        fig.suptitle(f"Task {tid}", fontsize=12, y=1.05)
        plot_arc_grid(axes[0], inp, "INPUT", show_gridlines)
        for i, (name, ok, err, pred) in enumerate(preds, start=1):
            if ok and pred is not None:
                plot_arc_grid(axes[i], pred, f"{name} PRED", show_gridlines)
            else:
                _plot_text_panel_3(axes[i], f"{name} PRED", f"ERROR\n{err}")
        plt.tight_layout()
        plt.show()
        return

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    fig.suptitle(f"Task {tid}", fontsize=12, y=1.02)

    plot_arc_grid(axes[0, 0], inp, "INPUT", show_gridlines)
    plot_arc_grid(axes[0, 1], gt, "GT", show_gridlines)
    _plot_text_panel_3(axes[0, 2], "Models", "gpt-4o\ngpt-5\ngpt-5-pro")

    for j, (name, ok, err, pred) in enumerate(preds):
        ax = axes[1, j]
        if ok and pred is not None:
            plot_arc_grid(ax, pred, f"{name} PRED", show_gridlines)
        else:
            _plot_text_panel_3(ax, f"{name} PRED", f"ERROR\n{err}")

    plt.tight_layout()
    plt.show()


# 예시: 첫 번째 task
visualize_input_gt_3models(df_vis, idx=0)

In [ ]:
# df_vis에 포함된 task들을 모두 4패널로 시각화
max_n = None  # 예: 10 (너무 많으면 제한)

n = len(df_vis) if max_n is None else min(int(max_n), len(df_vis))
for i in range(n):
    visualize_input_gt_3models(df_vis, idx=i)